# MiniMax M2 on Amazon Bedrock Mantle

MiniMax's M2 line on the `bedrock-mantle` endpoint. Three generations are live
simultaneously — more than any other family here — which makes it a good fit
for demonstrating a version-migration test.

**Models covered in this notebook**

| Model ID | Notes |
|---|---|
| `minimax.minimax-m2.5` | Latest generation |
| `minimax.minimax-m2.1` | Intermediate generation |
| `minimax.minimax-m2` | Original release |

## Which API? Chat Completions.
This family is served by the **OpenAI-compatible Chat Completions API** on the
`bedrock-mantle` endpoint, at the bare `/v1` path. The Responses API returns
**400 "does not support this API"** for these models — we prove that in §2 rather
than asking you to take it on trust.

## Self-contained, but see also
Everything you need is here. For deeper background on shared mechanics:
- **Auth (SigV4 (AWS Signature Version 4) + short-term API keys), the three URL paths,
  model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention / ZDR (zero data retention), CloudWatch
  namespace** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retry/backoff, service tiers, TTFT (time-to-first-token) measurement** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import err, parse_json_lenient, post, safe_print, ttft

REGION = "us-east-1"

M25 = "minimax.minimax-m2.5"
M21 = "minimax.minimax-m2.1"
M2 = "minimax.minimax-m2"


# Chat-Completions families live at the BARE /v1 path — not /openai/v1
# (that prefix is only for gemma-4, gpt-5.x and grok). See ../00-foundations/01.
PREFIX = "/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)
print("models  :", [M25, M21, M2])

base URL: https://bedrock-mantle.us-east-1.api.aws/v1
models  : ['minimax.minimax-m2.5', 'minimax.minimax-m2.1', 'minimax.minimax-m2']


## 1. First call

Auth is a short-term Bedrock API key minted from your ambient IAM credentials.
It expires within 12 hours and **cannot be refreshed** — mint a new one instead.
(`../00-foundations/01` shows the self-refreshing provider and the SigV4
alternative that needs no key at all.)

In [2]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Build the client from a FRESH token — don't construct one at import time and
# reuse it for hours, because the baked-in key expires.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

completion = client.chat.completions.create(
    model=M25,
    messages=[
        {
            "role": "user",
            "content": (
                "Explain what makes a model good at tool use, in two sentences."
            ),
        }
    ],
    # 600, not 250: this family reasons before answering, and a tight budget can
    # consume the whole allowance and return content=None (see the guard below).
    max_tokens=600,
)
# `content` can be None when the model spends the whole budget reasoning: the call
# succeeds with finish_reason="length" and no text. Check before printing -- this is
# the single most common surprise on this endpoint.
choice = completion.choices[0]
answer = choice.message.content or ""
if answer:
    print(answer)
else:
    print(f"(no text: finish_reason={choice.finish_reason!r} — raise max_tokens)")
print("\nusage:", completion.usage.model_dump_json())



A good tool‑use model accurately recognizes when a tool is needed, understands its interface and constraints, and can plan the correct sequence of actions to achieve the user's goal. Additionally, it effectively integrates the tool’s output back into its reasoning, adapting its plan based on feedback and handling errors robustly.

usage: {"completion_tokens":357,"prompt_tokens":52,"total_tokens":409,"completion_tokens_details":null,"prompt_tokens_details":null}


## 2. Why Chat Completions and not Responses

AWS recommends the Responses API for new applications in general — but
availability is per-model. Probe both surfaces so the 400 is visible:

In [3]:
for api_name, path, body in [
    (
        "Chat Completions",
        f"{PREFIX}/chat/completions",
        {
            "model": M25,
            "messages": [{"role": "user", "content": "Reply OK"}],
            "max_tokens": 16,
        },
    ),
    (
        "Responses (/v1)",
        f"{PREFIX}/responses",
        {"model": M25, "input": "Reply OK", "max_output_tokens": 16},
    ),
    (
        "Responses (/openai/v1)",
        "/openai/v1/responses",
        {"model": M25, "input": "Reply OK", "max_output_tokens": 16},
    ),
]:
    code, data = post(path, body, region=REGION)
    print(f"  {api_name:24} -> HTTP {code} {'' if code == 200 else err(data)[:64]}")

  Chat Completions         -> HTTP 200 


  Responses (/v1)          -> HTTP 400 The model 'minimax.minimax-m2.5' does not support the '/v1/respo


  Responses (/openai/v1)   -> HTTP 400 The model 'minimax.minimax-m2.5' does not support the '/openai/v


Concrete consequences of being Chat-Completions-only:

- **You own the conversation history.** There is no `previous_response_id`
  server-side state on this API — send the full `messages` array each turn.
- **Reasoning content is not returned.** `reasoning_effort` is accepted and the
  model does think, but the OpenAI Chat Completions schema has nowhere to put the
  trace, so you pay for those tokens without seeing them.
- Structured output uses `response_format`, not `text.format`.

## 3. Sampling parameters

This family accepts both `temperature` and `top_p`. That is *not* universal on
mantle — Gemma 4 rejects `top_p`, and Grok rejects `temperature` — so never share
one sampling config across families.

In [4]:
for label, extra in [
    ("temperature=0.7", {"temperature": 0.7}),
    ("temperature=0.0", {"temperature": 0.0}),
    ("top_p=0.95", {"top_p": 0.95}),
    ("both", {"temperature": 0.7, "top_p": 0.95}),
    ("max_tokens=1", {"max_tokens": 1}),
]:
    body = {
        "model": M25,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "max_tokens": 16,
    }
    body.update(extra)
    code, data = post(f"{PREFIX}/chat/completions", body, region=REGION)
    print(f"  {label:18} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  temperature=0.7    -> HTTP 200 


  temperature=0.0    -> HTTP 200 


  top_p=0.95         -> HTTP 200 


  both               -> HTTP 200 


  max_tokens=1       -> HTTP 200 


Note `max_tokens=1` is accepted here. The Responses API enforces a minimum of
16 — another reason the two surfaces are not interchangeable.

## 4. Streaming

Chat Completions streams `data: {...}` SSE (server-sent events) frames carrying
`choices[0].delta.content`, terminated by `data: [DONE]`.

In [5]:
stream = client.chat.completions.create(
    model=M25,
    messages=[
        {
            "role": "user",
            "content": (
                "List four signs that an agent's tool schema is badly designed."
            ),
        }
    ],
    max_tokens=300,
    stream=True,
)
chunks = 0
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        chunks += 1
        print(delta, end="", flush=True)
print(f"\n\n[{chunks} content deltas received]")



Here are four red‑flags that usually indicate an agent’s

 tool schema is poorly designed:

1. **Excessively deep or verbose

 nesting** – If a simple operation requires digging through five‑plus levels of

 objects, the schema is likely



[4 content deltas received]


## 5. Multi-turn — you manage the history

No server-side state on this API. Append each turn yourself.

In [6]:
messages = [
    {"role": "system", "content": "You are concise. Two sentences maximum."},
    {"role": "user", "content": "Why do agents benefit from typed tool schemas?"},
]
first = client.chat.completions.create(model=M25, messages=messages, max_tokens=200)
print("assistant:", first.choices[0].message.content)

messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append(
    {"role": "user", "content": "What happens when the schema is too loose?"}
)

second = client.chat.completions.create(model=M25, messages=messages, max_tokens=200)
print("\nassistant:", second.choices[0].message.content)
print(
    f"\ninput tokens grew: {first.usage.prompt_tokens} -> {second.usage.prompt_tokens}"
)

assistant: 

Typed tool schemas provide agents with clear input/output contracts, enabling:

1. **Better code generation** — LLMs can generate valid function calls with correct argument types
2. **Early validation** — errors caught before execution rather than at runtime
3. **Self-documenting interfaces** — agents understand what each tool expects and returns
4. **Improved error handling**



assistant: 

When schemas are too loose:

1. **Runtime failures** — LLMs generate plausible-but-wrong arguments that fail when executed
2. **Poor reasoning** — agents can't reliably predict tool behavior or validate inputs mentally
3. **Debugging difficulty** — errors surface late, making it hard to determine if the issue is the LLM, schema, or tool itself

Essentially, loose schemas shift errors from compile

input tokens grew: 33 -> 126


That growth is the cost of client-side history. Families on the Responses API can
avoid it with `previous_response_id` (see `../03-google-gemma/`), at the price of
30-day server-side retention.

## 6. Reasoning effort

`reasoning_effort` is accepted. The trace is not returned — but the token count
moves, which is how you can tell the model really is thinking harder.

In [7]:
print(f"{'effort':10} {'status':>7} {'completion tokens':>18}")
print("-" * 38)
for effort in ("none", "low", "medium", "high"):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": M25,
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "A car travels 60km at 30km/h then 60km at 60km/h. What is the "
                        "average speed? Show your working."
                    ),
                }
            ],
            "max_tokens": 400,
            "reasoning_effort": effort,
        },
        region=REGION,
    )
    tokens = (data.get("usage") or {}).get("completion_tokens", "-")
    print(f"  {effort:8} {code:>7} {tokens!s:>18}")

effort      status  completion tokens
--------------------------------------


  none         200                400


  low          200                400


  medium       200                400


  high         200                400


In [8]:
# Version migration: run one fixed prompt + schema across all three
# generations and diff the results. This is the check to run before
# switching a production model ID.
MIGRATION_SCHEMA = {
    "type": "object",
    "properties": {
        "average_speed_kmh": {"type": "number"},
        "method": {"type": "string"},
    },
    "required": ["average_speed_kmh", "method"],
    "additionalProperties": False,
}
PROMPT = (
    "A car drives 60km at 30km/h then 60km at 60km/h. "
    "What is the average speed over the whole trip?"
)

print(f"{'model':28} {'answer':>10}  method")
print("-" * 78)
for model in (M2, M21, M25):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": PROMPT}],
            "max_tokens": 400,
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "speed",
                    "strict": True,
                    "schema": MIGRATION_SCHEMA,
                },
            },
        },
        region=REGION,
    )
    if code != 200:
        print(f"{model:28} HTTP {code} {err(data)[:40]}")
        continue
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content") or ""
    # HTTP 200 does not mean you got an answer: check the finish reason first.
    if choice.get("finish_reason") == "length":
        print(
            f"{model:28} {'truncated':>10}  budget exhausted before the object closed"
        )
        continue
    try:
        got = parse_json_lenient(content)
    except ValueError as exc:
        print(f"{model:28} {'unparseable':>10}  {exc}"[:100])
        continue
    # A syntactically valid object can still be EMPTY. `.get()` would print
    # "None" and quietly look like an answer, so validate the keys.
    missing = {"average_speed_kmh", "method"} - set(got)
    if missing:
        print(f"{model:28} {'no data':>10}  parsed but missing {sorted(missing)}")
        continue
    print(f"{model:28} {got['average_speed_kmh']!s:>10}  {str(got['method'])[:40]}")
print("\n(The correct answer is 40 km/h — harmonic, not arithmetic, mean.)")
print("Read the failures as results too: this is exactly the pre-migration check.")

model                            answer  method
------------------------------------------------------------------------------


minimax.minimax-m2            truncated  budget exhausted before the object closed


minimax.minimax-m2.1          truncated  budget exhausted before the object closed


minimax.minimax-m2.5          truncated  budget exhausted before the object closed

(The correct answer is 40 km/h — harmonic, not arithmetic, mean.)
Read the failures as results too: this is exactly the pre-migration check.


### What that table is telling you

The failures are the point — this is what a migration check is *for*.

**Your table will not match this one**, and that is the finding. Across repeated
runs at `max_tokens=400` the older generations behave differently each time:

| Behaviour observed on the older generations | Why it matters |
|---|---|
| `finish_reason="length"` with **empty** `content` | HTTP 200, nothing usable |
| A valid object filled with **whitespace only** — no keys | `strict: True` passed, yet zero information |
| The correct answer, formatted oddly (comma at line start) | Valid JSON your parser must still accept |

`minimax-m2.5` answered correctly and cleanly on every run. The older two are the
unstable ones, and the *instability itself* is the reason to run this check before
changing a model ID.

Three lessons a reader should take away:

1. **`strict: True` does not guarantee a populated object.** An older generation
   returned `{}` — schema-valid, information-free. Validate the *keys you need*,
   not just that parsing succeeded.
2. **`.get()` hides this.** `got.get("average_speed_kmh")` on an empty dict prints
   `None`, which reads like an answer in a table. Check for missing keys explicitly,
   as the cell above now does.
3. **Newer is not automatically compatible, and older is not automatically safe.**
   Run this table before you change a model ID in production.

## 7. Tool use (function calling)

Chat Completions nests the schema under `"function"` — unlike the Responses API,
which puts `name`/`parameters` at the top level. Same concept, different shape.

In [9]:
def lookup_inventory(sku: str, warehouse: str = "main") -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0, "C-300": 7}
    return {
        "sku": sku,
        "warehouse": warehouse,
        "quantity": stock.get(sku.upper(), 0),
        "in_stock": stock.get(sku.upper(), 0) > 0,
    }


tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_inventory",
            "description": "Look up stock level for a SKU.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sku": {"type": "string", "description": "SKU code, e.g. A-100"},
                    "warehouse": {"type": "string", "enum": ["main", "overflow"]},
                },
                "required": ["sku"],
            },
        },
    }
]

convo = [{"role": "user", "content": "Do we have SKU A-100 in stock?"}]

# `tool_choice="auto"` means the model MAY call a tool -- not that it will. This
# family sometimes spends the whole budget reasoning and returns
# finish_reason="length" with no tool_calls at all (observed on ~1 call in 3 at
# some budgets). Retry rather than assuming the first response has a call.
msg = None
for attempt in range(1, 4):
    first = client.chat.completions.create(
        model=M25, messages=convo, tools=tools, tool_choice="auto", max_tokens=1500
    )
    choice = first.choices[0]
    print(
        f"attempt {attempt}: finish_reason={choice.finish_reason!r} "
        f"tool_calls={len(choice.message.tool_calls or [])}"
    )
    if choice.message.tool_calls:
        msg = choice.message
        break
if msg is None:
    raise RuntimeError("no tool call after 3 attempts - raise max_tokens further")

print(
    "tool_calls:",
    [(c.function.name, c.function.arguments) for c in (msg.tool_calls or [])],
)

if msg.tool_calls:
    convo.append(msg.model_dump(exclude_none=True))
    for call in msg.tool_calls:
        args = parse_json_lenient(call.function.arguments)
        result = lookup_inventory(**args)
        convo.append(
            {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)}
        )
    final = client.chat.completions.create(
        model=M25, messages=convo, tools=tools, max_tokens=200
    )
    print("\nfinal answer:", final.choices[0].message.content)

attempt 1: finish_reason='tool_calls' tool_calls=2
tool_calls: [('lookup_inventory', '{"sku": "A-100", "warehouse": "main"}'), ('lookup_inventory', '{"sku": "A-100", "warehouse": "overflow"}')]



final answer: 

Yes, SKU A-100 is in stock! We have:

- **Main warehouse:** 42 units
- **Overflow warehouse:** 42 units

That's a total of 84 units available across both warehouses. Would you like to place an order or need any other information?


### Forcing a specific tool

`tool_choice` can compel a named function. This is the most portable route to
strict structured output: the arguments *are* your JSON.

**But treat it as best-effort, not a guarantee.** In repeated testing about 1
call in 10 ignored the forced choice and returned prose with
`finish_reason="stop"`. Always check for the tool call and retry.

In [10]:
emit = [
    {
        "type": "function",
        "function": {
            "name": "emit_review",
            "description": "Return the structured review analysis.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "sentiment": {
                        "type": "string",
                        "enum": ["positive", "neutral", "negative"],
                    },
                    "would_recommend": {"type": "boolean"},
                },
                "required": ["summary", "sentiment", "would_recommend"],
            },
        },
    }
]


def emit_review(prompt, model=M25, attempts=3):
    """Forced tool call, with a retry.

    IMPORTANT: forcing `tool_choice` is honoured *almost* always, not always.
    In repeated testing roughly 1 call in 10 came back with finish_reason="stop"
    and prose instead of a tool call. Production code must handle that, so this
    helper retries rather than indexing [0] and hoping.
    """
    for attempt in range(attempts):
        completion = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            tools=emit,
            tool_choice={"type": "function", "function": {"name": "emit_review"}},
            max_tokens=300,
        )
        choice = completion.choices[0]
        calls = choice.message.tool_calls or []
        if calls:
            if attempt:
                print(f"(succeeded on attempt {attempt + 1})")
            # parse_json_lenient, not json.loads: some models append characters
            # after a well-formed object even in strict modes.
            return parse_json_lenient(calls[0].function.arguments)
        print(
            f"attempt {attempt + 1}: no tool call "
            f"(finish_reason={choice.finish_reason}) — retrying"
        )
    raise RuntimeError("model would not emit the forced tool call")


review = emit_review("Review: 'Fast delivery, but the packaging arrived crushed.'")
print(json.dumps(review, indent=2))

{
  "summary": "Fast delivery, but the packaging arrived crushed.",
  "sentiment": "neutral",
  "would_recommend": false
}


## 8. Structured output with `response_format`

Two variants: loose `json_object`, and schema-enforced `json_schema`.

### Budget enough tokens, or you get nothing

A reasoning-capable model may spend most of its budget thinking before it emits
the opening brace. If `max_tokens` runs out first you get **HTTP 200 with empty
content** and `finish_reason="length"` - not an error, just nothing usable.
Always check `finish_reason` before parsing.

In [11]:
def json_object_call(prompt, max_tokens, model=M25):
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "response_format": {"type": "json_object"},
        },
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content") or ""
    return code, choice.get("finish_reason"), content


PROMPT = "Give the capital and population of France as JSON."
for budget in (64, 600):
    code, finish, content = json_object_call(PROMPT, budget)
    print(
        f"max_tokens={budget:4} HTTP {code} finish={finish!s:8} "
        f"content_len={len(content)}"
    )
    if finish == "length" and not content.strip():
        print("    -> truncated before any JSON was emitted; raise max_tokens")
    elif content.strip():
        print("    ->", parse_json_lenient(content))

max_tokens=  64 HTTP 200 finish=length   content_len=0
    -> truncated before any JSON was emitted; raise max_tokens


max_tokens= 600 HTTP 200 finish=stop     content_len=73
    -> {'country': 'France', 'capital': 'Paris', 'population': 68000000}


In [12]:
schema = {
    "type": "object",
    "properties": {
        "country": {"type": "string"},
        "capital": {"type": "string"},
        "population_millions": {"type": "number"},
    },
    "required": ["country", "capital", "population_millions"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": M25,
        "messages": [{"role": "user", "content": "Describe France."}],
        "max_tokens": 250,
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "country", "strict": True, "schema": schema},
        },
    },
    region=REGION,
)
choice = (data.get("choices") or [{}])[0]
content = choice.get("message", {}).get("content")  # may be None!
print("json_schema ->", code, "| finish_reason:", choice.get("finish_reason"))
print("raw:", repr((content or "")[:160]))

if choice.get("finish_reason") == "length":
    # Reasoning consumed the budget before the object closed. Retry bigger.
    print("truncated - retrying with a larger budget")
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": M25,
            "messages": [{"role": "user", "content": "Describe France."}],
            "max_tokens": 2000,
            "response_format": {
                "type": "json_schema",
                "json_schema": {"name": "country", "strict": True, "schema": schema},
            },
        },
        region=REGION,
    )
    choice = (data.get("choices") or [{}])[0]
    content = choice.get("message", {}).get("content")
    print("retry finish_reason:", choice.get("finish_reason"))

parsed = parse_json_lenient(content or "")
print("parsed:", json.dumps(parsed, indent=2))
missing = {"country", "capital"} - set(parsed)
if missing:
    raise ValueError(f"model omitted required keys: {sorted(missing)} in {parsed}")
print("required keys present: country, capital")

json_schema -> 200 | finish_reason: stop
raw: '{\n\n"country": "France"\n, "capital": "Paris"\n, "population_millions": 68\n    }'
parsed: {
  "country": "France",
  "capital": "Paris",
  "population_millions": 68
}
required keys present: country, capital


**Always parse leniently.** Even in strict mode, some mantle models append
characters after a valid object (Gemma 4 does this in ~half of runs), which makes
a bare `json.loads()` raise on output that is otherwise fine.

## 9. Compare the models in this family

All three generations on one prompt — the migration diff.

In [13]:
task = "In one sentence, why is a harmonic mean the right average for speeds?"

print(f"{'model':44} {'latency':>9} {'out tok':>8}  answer")
print("-" * 108)
for model in [M2, M21, M25]:
    started = time.perf_counter()
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": model,
            "messages": [{"role": "user", "content": task}],
            "max_tokens": 160,
        },
        region=REGION,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:44} {'-':>9} {'-':>8}  HTTP {code}: {err(data)[:40]}")
        continue
    text = (data["choices"][0]["message"]["content"] or "").strip().replace("\n", " ")
    print(
        f"{model:44} {elapsed:>8.2f}s "
        f"{data['usage']['completion_tokens']:>8}  {text[:44]!r}"
    )

model                                          latency  out tok  answer
------------------------------------------------------------------------------------------------------------


minimax.minimax-m2                               2.24s      160  ''


minimax.minimax-m2.1                             4.10s      160  ''


minimax.minimax-m2.5                             3.12s      160  ''


## 10. Latency: TTFT and throughput

TTFT is dominated by *prefill* (the model reading your prompt) plus queue time.
Service tiers trade cost against queue priority — they mostly separate under
contention, so single samples on an idle account look flat.
(`../00-foundations/03` has the full treatment.)

In [14]:
print(f"{'tier':10} {'TTFT (s)':>10} {'total (s)':>10} {'frames/s':>10}")
print("-" * 44)
for tier in ("default", "flex", "priority"):
    m = ttft(
        f"{PREFIX}/chat/completions",
        {
            "model": M25,
            "messages": [
                {
                    "role": "user",
                    "content": (
                        "List four signs that an agent's tool schema is badly designed."
                    ),
                }
            ],
            "max_tokens": 200,
            "service_tier": tier,
        },
        region=REGION,
    )
    if m.get("error"):
        print(f"{tier:10} {m['error']:>32}  (tier not supported by this model)")
    else:
        print(
            f"{tier:10} {m['ttft_s']:>10.3f} {m['total_s']:>10.3f} "
            f"{m['frames_per_s']:>10.1f}"
        )

tier         TTFT (s)  total (s)   frames/s
--------------------------------------------


default         1.483      4.665        5.7


flex            2.881      3.648        3.9


priority       90.814     95.815        4.6


## 11. Production hardening

Retries, cost attribution, and privacy. Mantle has **no RPM quota** — throttling
is token-based, and most models here have no published TPM (tokens per minute) quota
at all, so
capacity is fair-share. That makes retry-with-backoff mandatory, not optional.

In [15]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "minimax-samples",
        "tags": {"Application": "MiniMaxDemo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)

code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": M25,
        "messages": [{"role": "user", "content": "Reply OK"}],
        "max_tokens": 16,
    },
    region=REGION,
    headers={"OpenAI-Project": project_id},  # cost attribution
)
print("attributed call ->", code)

project: 200 proj_onjdo7h4...


attributed call -> 200


In [16]:
class MiniMaxClient:
    """Demonstrates retry, attribution and structured-output patterns for this
    family on bedrock-mantle. A teaching pattern, not a production component:
    review and adapt it, and have it security-reviewed, before deployment."""

    def __init__(self, model=M25, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = model, region, tier, project

    def chat(self, messages, *, max_tokens=512, tools=None, schema=None, effort=None):
        body = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0.7,
            "service_tier": self.tier,
        }
        if tools:
            body["tools"] = tools
        if effort:
            body["reasoning_effort"] = effort
        if schema:
            body["response_format"] = {
                "type": "json_schema",
                "json_schema": {"name": "out", "strict": True, "schema": schema},
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429 + 5xx with exponential backoff and jitter.
        code, data = post(
            f"{PREFIX}/chat/completions", body, region=self.region, headers=headers
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data

    @staticmethod
    def _choice(data: dict) -> dict:
        """First choice, without assuming the list is non-empty."""
        return (data.get("choices") or [{}])[0]

    def json(self, prompt, schema, *, max_tokens=512, **kw):
        """Structured call that survives a truncated first attempt.

        This applies §8's lesson rather than restating it: a reasoning model can
        spend the whole budget thinking and return HTTP 200 with EMPTY content and
        finish_reason="length". Parsing that raises. So check finish_reason, and
        escalate the budget once before giving up.
        """
        messages = [{"role": "user", "content": prompt}]
        for budget in (max_tokens, max_tokens * 4):
            data = self.chat(messages, schema=schema, max_tokens=budget, **kw)
            choice = self._choice(data)
            content = choice.get("message", {}).get("content") or ""
            if content.strip():
                return parse_json_lenient(content)
            if choice.get("finish_reason") != "length":
                break  # empty for some other reason - escalating will not help
        raise RuntimeError(
            f"no content after budget escalation to {max_tokens * 4} tokens "
            f"(finish_reason={self._choice(data).get('finish_reason')!r})"
        )


bot = MiniMaxClient(tier="flex", project=project_id)
out = bot.json(
    "Name the largest ocean and its average depth in metres.",
    {
        "type": "object",
        "properties": {"ocean": {"type": "string"}, "avg_depth_m": {"type": "number"}},
        "required": ["ocean", "avg_depth_m"],
        "additionalProperties": False,
    },
)
print("structured result:", out)

structured result: {'ocean': 'Pacific Ocean', 'avg_depth_m': 4280}


In [17]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived demo project:", code, archived.get("status"))

archived demo project: 200 archived


## Gotchas — MiniMax M2 on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Path prefix | Bare `/v1`, **not** `/openai/v1` (that's gemma-4 / gpt-5.x / grok) |
| Responses API | Returns **400** for this family — Chat Completions only |
| History | No `previous_response_id`; you send `messages` every turn |
| Reasoning trace | `reasoning_effort` works but the trace is never returned |
| Strict JSON | Parse leniently — models can append text after a valid object |
| Sampling | `temperature` **and** `top_p` both fine here; not true family-wide |
| `max_tokens` | 1 is valid here; Responses API demands ≥16 |
| `content` can be `None` | Check `finish_reason` before slicing/parsing content |
| Quotas | No RPM quota; most models have no published TPM — retry with backoff |
| `reserved` tier | Rejected as a parameter; arranged via your account team |
| CloudWatch | Metrics land in `AWS/BedrockMantle`, not `AWS/Bedrock` |
| Three live generations | Pin an exact model ID in production |
| Agentic focus | M2 is tuned for tool use — lean on typed schemas |

## Where next
- Same API shape: `../04-qwen/`, `../06-zai-glm/`, `../08-moonshot-kimi/`
- Different API shape: `../03-google-gemma/` (Responses),
  `../02-anthropic-claude/` (Messages), `../01-openai-gpt/` (web search, caching)
- Shared mechanics: `../00-foundations/`

## Also on `bedrock-runtime`? MiniMax

All three generations are on both endpoints under the same IDs, so the version-migration check earlier in this notebook can be repeated on Converse.

`endpoints_for()` asks both catalogues rather than trusting a table, so the cell
below tells you today's answer. Converse is worth reaching for when you want one
request shape across providers, or a feature that only `bedrock-runtime` carries.


In [18]:
from bedrock import (
    converse,
    converse_reasoning,
    endpoints_for,
    resolve_runtime_id,
)

MANTLE_ID = "minimax.minimax-m2.5"
RUNTIME_ID = "minimax.minimax-m2.5"

print("endpoint availability:", endpoints_for(MANTLE_ID))
print("mantle model id :", MANTLE_ID)
print("runtime model id:", RUNTIME_ID)
resolved = resolve_runtime_id(RUNTIME_ID)
print("converse sends  :", resolved)
if resolved != RUNTIME_ID:
    print("                  ^ resolved for you; the form above would be rejected")

# The same question, through Converse. Note the shape: content is a LIST of
# blocks rather than a string, and the token budget lives in inferenceConfig.
# The budget is generous on purpose - a reasoning model spends it on the trace
# first and returns no text block at all if it runs out.
text, response = converse(
    RUNTIME_ID,
    [
        {
            "role": "user",
            "content": [
                {"text": "Name one benefit of idempotency. One sentence."}
            ],
        }
    ],
    max_tokens=400,
    system="You are terse.",
)

error = (response.get("error") or {}).get("message")
if error:
    print("\ncall failed:", error[:200])
else:
    reasoning = converse_reasoning(response)
    print("\nstop reason:", response.get("stopReason"))
    print("tokens     :", response.get("usage", {}).get("totalTokens"))
    if reasoning:
        print(f"reasoning  : {len(reasoning)} chars (returned in a "
              "reasoningContent block, before the text)")
    if text.strip():
        print("answer     :", text.strip()[:200])
    else:
        # Empty text is NOT the same as a failed call. Say which it is.
        print("answer     : (none - the budget went to reasoning; raise max_tokens)")


endpoint availability: {'mantle': True, 'runtime': True}
mantle model id : minimax.minimax-m2.5
runtime model id: minimax.minimax-m2.5


converse sends  : minimax.minimax-m2.5



stop reason: end_turn
tokens     : 206
reasoning  : 756 chars (returned in a reasoningContent block, before the text)
answer     : Idempotency ensures that repeating the same operation yields the same result, making systems more reliable and predictable.
